# Gift Recommendation LLM Evaluation

This notebook evaluates different LLM models for gift recommendation generation.

## Metrics Tracked
- **latency_ms**: Time per prompt
- **tokens_in / tokens_out**: Inference cost basis
- **quality_score**: LLM-as-a-judge or rubric scoring
- **monthly_estimate**: Cost for 2B requests

## Models Evaluated
- OpenAI compatible API (e.g., OpenAI, Token Factory)
- Self-hosted vLLM (7B, 13B)

In [ ]:
%load_ext autoreload
%autoreload 2


import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "prototype" / "src"))

from src.clients.llm_client import LLMClient
from src.evaluation import (
    evaluate_quality_with_judge,
    setup_mlflow,
)
from src.generators import generate_gift_recommendation
from src.prompts import (
    generate_gift_quality_judge_prompt,
)
from src.utils import load_env_from_repo_root

# Load .env file from repository root
load_env_from_repo_root('.env', override=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [37]:
setup_mlflow("gift_recommendation_eval")

MLflow tracking URI: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud
MLflow experiment: <Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1765790976980, experiment_id='1', last_update_time=1765790976980, lifecycle_stage='active', name='gift_recommendation_eval', tags={}>


In [50]:
models_to_evaluate = []

# OpenAI models (publicly available as of 2025)
if os.getenv("OPENAI_API_KEY"):
    models_to_evaluate.append({
        "name": "gpt-4o-mini",
        "client": LLMClient(
            base_url="https://api.openai.com/v1",
            api_key=os.getenv("OPENAI_API_KEY"),
            model="gpt-4o-mini"
        ),
        "cost_per_1m_tokens_in": 0.15,
        "cost_per_1m_tokens_out": 0.6,
    })
    # models_to_evaluate.append({
    #     "name": "gpt-3.5-turbo",
    #     "client": LLMClient(
    #         base_url="https://api.openai.com/v1",
    #         api_key=os.getenv("OPENAI_API_KEY"),
    #         model="gpt-3.5-turbo"
    #     ),
    #     "cost_per_1m_tokens_in": 1.5,
    #     "cost_per_1m_tokens_out": 2.0,
    # })

# Token Factory models (if configured)
if os.getenv("TOKEN_FACTORY_API_KEY") and os.getenv("TOKEN_FACTORY_BASE_URL"):
    models_to_evaluate.append({
        "name": "tf/gpt-oss-20b",
        "client": LLMClient(
            base_url=os.getenv("TOKEN_FACTORY_BASE_URL"),
            api_key=os.getenv("TOKEN_FACTORY_API_KEY"),
            model="openai/gpt-oss-20b"
        ),
        "cost_per_1m_tokens_in": 0.15,
        "cost_per_1m_tokens_out": 0.6,
    })
    models_to_evaluate.append({
        "name": "tf/DeepSeek-R1-0528",
        "client": LLMClient(
            base_url=os.getenv("TOKEN_FACTORY_BASE_URL"),
            api_key=os.getenv("TOKEN_FACTORY_API_KEY"),
            model="deepseek-ai/DeepSeek-R1-0528"
        ),
        "cost_per_1m_tokens_in": 0.8,
        "cost_per_1m_tokens_out": 2.4,
    })

# vllm_base_url = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
# if vllm_base_url:
#     models_to_evaluate.append({
#         "name": "vllm-llama-2-7b",
#         "client": LLMClient(
#             base_url=vllm_base_url,
#             api_key=None,
#             model=os.getenv("VLLM_MODEL_7B", "meta-llama/Llama-2-7b-chat-hf")
#         ),
#         "cost_per_1m_tokens_in": 0.0,
#         "cost_per_1m_tokens_out": 0.0,
#     })
#     models_to_evaluate.append({
#         "name": "vllm-llama-2-13b",
#         "client": LLMClient(
#             base_url=vllm_base_url,
#             api_key=None,
#             model=os.getenv("VLLM_MODEL_13B", "meta-llama/Llama-2-13b-chat-hf")
#         ),
#         "cost_per_1m_tokens_in": 0.0,
#         "cost_per_1m_tokens_out": 0.0,
#     })

In [ ]:
print("Smoke-testing model availability...")

for m in models_to_evaluate:
    model_name = m["name"]
    try:
        text, metrics = m["client"].generate(
            prompt="ping",
            temperature=0.0,
            max_tokens=8,
        )
        print(f"✅ {model_name}: reachable (latency_ms={metrics.get('latency_ms'):.1f}, tokens_out={metrics.get('tokens_out')})")
    except Exception as e:
        print(f"❌ {model_name}: {e}")

Smoke-testing model availability...
✅ gpt-4o-mini: reachable (latency_ms=772.7, tokens_out=8)
✅ tf/gpt-oss-20b: reachable (latency_ms=203.7, tokens_out=8)
✅ tf/DeepSeek-R1-0528: reachable (latency_ms=459.3, tokens_out=8)


### Judge Client - gpt-5.2

In [ ]:
judge_client = None


judge_client = LLMClient(
    base_url="https://api.openai.com/v1",
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-5.2"
)


In [44]:
# # test_prompt = generate_gift_recommendation_prompt(test_profiles[0])
# # test_response, _ = client.generate(test_prompt, temperature=0.8, max_tokens=300)

# # # judge_response, _ = judge_client.generate(test_judge_prompt, temperature=0.0)
# # test_judge_prompt = generate_gift_quality_judge_prompt(test_profiles[0], test_response)
# quality_score = evaluate_quality_with_judge(judge_client, test_judge_prompt)
# quality_score

## Evaluate Gift Recommendations

In [ ]:
import re
from pathlib import Path

import mlflow
import pandas as pd

results = []
agg_results = []

# Enable automatic tracing for all OpenAI API calls.
MLFLOW_AVAILABLE = bool(mlflow.get_tracking_uri())
mlflow.openai.autolog()

# Evaluate all configured models
for model_config in models_to_evaluate:
    model_name = model_config["name"]
    client = model_config["client"]
    print(f"Evaluating model: {model_name}")

    with mlflow.start_run(run_name=f"{model_name}"):
        parent_run_id = mlflow.active_run().info.run_id if MLFLOW_AVAILABLE else None
        per_calls = []

        for kid_profile in test_profiles:
            try:
                gift_recommendation, metrics = generate_gift_recommendation(
                    client, kid_profile, temperature=0.0, max_tokens=300
                )

                # Format gift_recommendation as string for judge evaluation
                response_text = "GIFTS:\n" + "\n".join(f"- {gift}" for gift in gift_recommendation.gifts)
                response_text += f"\n\nRATIONALE:\n{gift_recommendation.rationale}"

                quality_score = None
                if judge_client:
                    judge_prompt = generate_gift_quality_judge_prompt(kid_profile, response_text)
                    quality_score = evaluate_quality_with_judge(judge_client, judge_prompt)

                tokens_in = metrics.get("tokens_in", 0) or 0
                tokens_out = metrics.get("tokens_out", 0) or 0
                cost_in = round(tokens_in * model_config["cost_per_1m_tokens_in"], 1)
                cost_out = round(tokens_out * model_config["cost_per_1m_tokens_out"], 1)
                cost_total = round(cost_in + cost_out, 1)

                record = {
                    "model": model_name,
                    "kid_id": kid_profile.id,
                    "latency_ms": round(metrics.get("latency_ms", 0), 0),
                    "tokens_in": tokens_in,
                    "tokens_out": tokens_out,
                    "quality_score": quality_score,
                    "cost_in_1m": cost_in,
                    "cost_out_1m": cost_out,
                    "cost_total_1m": cost_total,
                }
                per_calls.append(record)
                results.append(record)

                if MLFLOW_AVAILABLE:
                    with mlflow.start_run(run_name=f"{kid_profile.id}", nested=True):
                        mlflow.log_metric("latency_ms", record["latency_ms"])
                        mlflow.log_metric("tokens_in", tokens_in)
                        mlflow.log_metric("tokens_out", tokens_out)
                        if quality_score is not None:
                            mlflow.log_metric("quality_score", quality_score)
                        mlflow.log_metric("cost_in_1m", cost_in)
                        mlflow.log_metric("cost_out_1m", cost_out)
                        mlflow.log_metric("cost_total_1m", cost_total)
                        mlflow.set_tags({
                            "kid_id": kid_profile.id,
                            "task": "gift_recommendation",
                        })
                        mlflow.log_text(response_text, "gift_recommendation.txt")

            except Exception as e:
                err_rec = {
                    "model": model_name,
                    "kid_id": kid_profile.id,
                    "error": str(e),
                }
                per_calls.append(err_rec)
                results.append(err_rec)

        df_model = pd.DataFrame([r for r in per_calls if "error" not in r])
        if not df_model.empty:
            agg = {
                "model": model_name,
                "latency_ms": round(df_model["latency_ms"].mean(), 0),
                "tokens_in": df_model["tokens_in"].mean(),
                "tokens_out": df_model["tokens_out"].mean(),
                "quality_score": df_model["quality_score"].mean(),
                "cost_in_1m": round(df_model["cost_in_1m"].mean(), 1),
                "cost_out_1m": round(df_model["cost_out_1m"].mean(), 1),
                "cost_total_1m": round(df_model["cost_total_1m"].mean(), 1),
                "calls": len(df_model),
            }
            agg_results.append(agg)

            out_dir = Path("data/evaluation")
            out_dir.mkdir(parents=True, exist_ok=True)
            model_name_path = re.sub(r'[^\w\-]', '-', model_name)
            out_path = out_dir / f"01_gift_rec_eval-{model_name_path}.csv"
            df_model.to_csv(out_path, index=False)
            print(f"Saved per-call results for {model_name} -> {out_path}")

            if MLFLOW_AVAILABLE:
                mlflow.log_metric("latency_ms", agg["latency_ms"])
                mlflow.log_metric("tokens_in", agg["tokens_in"])
                mlflow.log_metric("tokens_out", agg["tokens_out"])
                mlflow.log_metric("quality_score", agg["quality_score"])
                mlflow.log_metric("cost_in_1m", agg["cost_in_1m"])
                mlflow.log_metric("cost_out_1m", agg["cost_out_1m"])
                mlflow.log_metric("cost_total_1m", agg["cost_total_1m"])
                mlflow.log_metric("calls", agg["calls"])
                mlflow.set_tags({"task": "gift_recommendation"})
        else:
            print(f"No successful calls for model {model_name}")


Evaluating model: gpt-4o-mini
🏃 View run 450dd158-c0b4-4c0e-99f0-bc463bcd5078 at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/1/runs/122d22db84a24018ab73641883955389
🧪 View experiment at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/1
🏃 View run af0e9523-c90b-4c04-870d-bdf2724ca607 at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/1/runs/b394d18ce60f4f0585b7d858fa95f8dd
🧪 View experiment at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/1
🏃 View run 282a1616-a579-4e4c-bfd4-9148477b7d26 at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/1/runs/f931d818fb354159bc2cedac6611e7b2
🧪 View experiment at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw

In [60]:
# Final aggregated summary
df_results = pd.DataFrame(results)
df_agg = pd.DataFrame(agg_results)
print("Per-model aggregate summary:")
df_agg.round(3)

Per-model aggregate summary:


,model,latency_ms,tokens_in,tokens_out,quality_score,cost_in_1m,cost_out_1m,cost_total_1m,calls
0,gpt-4o-mini,2318.0,205.00,90.5,0.912,30.8,54.3,85.1,4
1,tf/gpt-oss-20b,1206.0,267.00,292.5,0.495,40.0,175.5,215.5,4
2,tf/DeepSeek-R1-0528,11656.0,210.25,300.0,0.312,168.2,720.0,888.2,4
